In [1]:
!pip -q install datasets transformers sentence-transformers faiss-cpu rank-bm25 evaluate accelerate scikit-learn gensim nltk gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 89.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 77.5 MB/s eta 0:00:00:00:0100:01


In [2]:
import os
import re
import gc
import json
import time
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder

from datasets import load_dataset, Dataset
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    pipeline
)

from sentence_transformers import SentenceTransformer, CrossEncoder

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss
import gradio as gr

from gensim.models import Word2Vec, FastText

warnings.filterwarnings('ignore')

In [3]:
for pkg in ["punkt", "stopwords", "wordnet", "omw-1.4"]:
    try:
        nltk.download(pkg, quiet=True)
    except:
        pass

## 1. Load Dataset

In [4]:
dataset = load_dataset(
    "PolyAI/banking77",
    revision="830c4f2be6949546b11fe83fbc50993348a2bccd"
)
dataset

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/295k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/93.0k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10003 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3080 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})

In [5]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_DIR = Path("./finassist_project")
PROJECT_DIR.mkdir(exist_ok=True, parents=True)
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True, parents=True)
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True, parents=True)
REPORT_DIR = PROJECT_DIR / "reports"
REPORT_DIR.mkdir(exist_ok=True, parents=True)

In [6]:
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

label_names = dataset["train"].features["label"].names
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in id2label.items()}

train_df["label_name"] = train_df["label"].map(id2label)
test_df["label_name"] = test_df["label"].map(id2label)

In [7]:
print(train_df.shape, test_df.shape)
train_df.head()

(10003, 3) (3080, 3)


,text,label,label_name
0,I am still waiting on my card?,11,card_arrival
1,What can I do if my card still hasn't arrived ...,11,card_arrival
2,I have been waiting over a week. Is the card s...,11,card_arrival
3,Can I track my card while it is in the process...,11,card_arrival
4,"How do I know if I will get my card, or if it ...",11,card_arrival


In [8]:
train_df["text_len_words"] = train_df["text"].str.split().str.len()
train_df["text_len_chars"] = train_df["text"].str.len()

display(train_df[["text_len_words", "text_len_chars"]].describe())
display(train_df["label_name"].value_counts().head(15))

,text_len_words,text_len_chars
count,10003.000000,10003.000000
mean,11.949415,59.473758
std,7.891577,40.867901
min,2.000000,13.000000
25%,7.000000,36.000000
50%,10.000000,47.000000
75%,13.000000,64.000000
max,79.000000,433.000000


,count
label_name,
card_payment_fee_charged,187
direct_debit_payment_not_recognised,182
balance_not_updated_after_cheque_or_cash_deposit,181
wrong_amount_of_cash_received,180
cash_withdrawal_charge,177
transaction_charged_twice,175
declined_cash_withdrawal,173
transfer_fee_charged,172
balance_not_updated_after_bank_transfer,171


## 2. Text preprocessing


In [9]:
STOPWORDS = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [ ]:
def basic_clean(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"@[A-Za-z0-9_]+", " <MENTION> ", text)
    text = re.sub(r"#[A-Za-z0-9_]+", " <HASHTAG> ", text)
    text = re.sub(r"\d+", " <NUMBER> ", text)
    text = re.sub(r"[^a-z<>\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def classical_preprocess(text : str) ->str:
    text = basic_clean(text)
    tokens = []
    for tok in text.split():
        if tok in STOPWORDS:
            continue
        if tok.startswith("<") and tok.endswith(">"):
            tokens.append(tok)
        else:
            tokens.append(lemmatizer.lemmatize(tok))
    return " ".join(tokens)

In [11]:
train_df["text_clean"] = train_df["text"].apply(classical_preprocess)
test_df["text_clean"] = test_df["text"].apply(classical_preprocess)

train_df[["text", "text_clean"]].head(10)

,text,text_clean
0,I am still waiting on my card?,still waiting card
1,What can I do if my card still hasn't arrived ...,card still arrived < > week
2,I have been waiting over a week. Is the card s...,waiting week card still coming
3,Can I track my card while it is in the process...,track card process delivery
4,"How do I know if I will get my card, or if it ...",know get card lost
5,When did you send me my new card?,send new card
6,Do you have info about the card on delivery?,info card delivery
7,What do I do if I still have not received my n...,still received new card
8,Does the package with my card have tracking?,package card tracking
9,I ordered my card but it still isn't here,ordered card still


## 3. Classical baseline: TF-IDF + Logistic Regression


In [12]:
tfidf_clf = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        max_features=50000
    )),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

tfidf_clf.fit(train_df["text_clean"], train_df["label_name"])
tfidf_preds = tfidf_clf.predict(test_df["text_clean"])

tfidf_acc = accuracy_score(test_df["label_name"], tfidf_preds)
tfidf_f1 = f1_score(test_df["label_name"], tfidf_preds, average="weighted")

print("TF-IDF Accuracy:", round(tfidf_acc, 4))
print("TF-IDF Weighted F1:", round(tfidf_f1, 4))

TF-IDF Accuracy: 0.8523
TF-IDF Weighted F1: 0.8526


## 4. Dense embeddings: Word2Vec and FastText

In [13]:
tokenized_sentences = [x.split() for x in train_df["text_clean"].tolist()]

w2v_model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    seed=SEED
)

ft_model = FastText(
    sentences=tokenized_sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    seed=SEED
)

print("FastText OOV-style vector:", ft_model.wv["virtualcard"][:5])

FastText OOV-style vector: [-0.1952327  -0.19901037  0.20326313  0.00211654 -0.00340311]


In [14]:
def average_embedding(text, model, dim=100):
    tokens = text.split()
    vecs = [model.wv[t] for t in tokens if t in model.wv]
    if len(vecs) == 0:
        return np.zeros(dim, dtype=np.float32)
    
    return np.mean(vecs, axis=0).astype(np.float32)

x_train_w2v = np.stack([average_embedding(t, w2v_model, 100) for t in train_df["text_clean"]])
x_test_w2v = np.stack([average_embedding(t, w2v_model, 100) for t in test_df["text_clean"]])

In [15]:
w2v_lr = LogisticRegression(max_iter=2000, class_weight='balanced')
w2v_lr.fit(x_train_w2v, train_df["label_name"])

w2v_preds = w2v_lr.predict(x_test_w2v)

w2v_acc = accuracy_score(test_df["label_name"], w2v_preds)
w2v_f1 = f1_score(test_df["label_name"], w2v_preds, average='weighted')

print("Word2Vec AvgEmb Accuracy:", round(w2v_acc, 4))
print("Word2Vec AvgEmb Weighted F1:", round(w2v_f1, 4))

Word2Vec AvgEmb Accuracy: 0.5831
Word2Vec AvgEmb Weighted F1: 0.5722


## 5. BiLSTM baseline

In [16]:
MAX_VOCAB = 20000
MAX_LEN = 40
EMBED_DIM = 100
HIDDEN_DIM = 128
BATCH_SIZE = 128
EPOCHS_LSTM = 8
DEVICE = "cuda"

In [17]:
label_encoder = LabelEncoder()
word_counter = Counter()

y_train_enc = label_encoder.fit_transform(train_df["label_name"])
y_test_enc = label_encoder.transform(test_df["label_name"])

In [18]:
for text in train_df["text_clean"]:
    word_counter.update(text.split())
    
special_tokens = ["<PAD>", "<UNK>"]
vocab_words = [w for w, _ in word_counter.most_common(MAX_VOCAB - len(special_tokens))]
vocab = {w: i for i, w in enumerate(special_tokens + vocab_words)}

pad_id = vocab["<PAD>"]
unk_id = vocab["<UNK>"]

In [19]:
def encode_text(text, max_len=MAX_LEN):
    ids = [vocab.get(tok, unk_id) for tok in text.split()][:max_len]
    if len(ids) < max_len:
        ids += [pad_id] * (max_len - len(ids))
    return ids

x_train_ids = np.array([encode_text(t) for t in train_df["text_clean"]], dtype=np.int64)
x_test_ids = np.array([encode_text(t) for t in test_df["text_clean"]], dtype=np.int64)

In [20]:
from torch.utils.data import Dataset as TorchDataset, DataLoader

class TextDataset(TorchDataset):
    def __init__(self, x, y):
        self.x = torch.tensor(x, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]
    
train_loader = DataLoader(TextDataset(x_train_ids, y_train_enc), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TextDataset(x_test_ids, y_test_enc), batch_size=BATCH_SIZE, shuffle=False)

In [21]:
embedding_matrix = np.random.normal(0, 0.1, size=(len(vocab), EMBED_DIM)).astype(np.float32)
embedding_matrix[pad_id] = 0

for word, idx in vocab.items():
    if word in ft_model.wv:
        embedding_matrix[idx] = ft_model.wv[word]

In [22]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, embedding_matrix=None):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        if embedding_matrix is not None:
            self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
    
    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        h = torch.cat([h_n[-2], h_n[-1]], dim=1)
        h = self.dropout(h)
        return self.fc(h)

In [23]:
criterion = nn.CrossEntropyLoss()

lstm_model = BiLSTMClassifier(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=len(label_encoder.classes_),
    embedding_matrix=embedding_matrix
).to(DEVICE)

optimizer = torch.optim.AdamW(lstm_model.parameters(), lr=2e-4)

In [24]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    losses, preds_all, true_all = [], [], []
    for xb, yb in tqdm(loader, leave=False):
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        preds_all.extend(logits.argmax(dim=1).detach().cpu().numpy())
        true_all.extend(yb.detach().cpu().numpy())
    return np.mean(losses), accuracy_score(true_all, preds_all), f1_score(true_all, preds_all, average="weighted")

In [25]:
@torch.no_grad()
def eval_model(model, loader, criterion, device):
    model.eval()
    losses, preds_all, true_all = [], [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        losses.append(loss.item())
        preds_all.extend(logits.argmax(dim=1).cpu().numpy())
        true_all.extend(yb.cpu().numpy())
    return np.mean(losses), accuracy_score(true_all, preds_all), f1_score(true_all, preds_all, average="weighted"), preds_all

In [26]:
for epoch in range(EPOCHS_LSTM):
    tr_loss, tr_acc, tr_f1 = train_one_epoch(lstm_model, train_loader, optimizer, criterion, DEVICE)
    va_loss, va_acc, va_f1, _ = eval_model(lstm_model, test_loader, criterion, DEVICE)
    print(f"Epoch {epoch+1}/{EPOCHS_LSTM} | train_acc={tr_acc:.4f} | val_acc={va_acc:.4f} | val_f1={va_f1:.4f}")

  0%|          | 0/79 [00:00<?, ?it/s]

Epoch 1/8 | train_acc=0.0159 | val_acc=0.0140 | val_f1=0.0011


  0%|          | 0/79 [00:00<?, ?it/s]

Epoch 2/8 | train_acc=0.0192 | val_acc=0.0224 | val_f1=0.0017


  0%|          | 0/79 [00:00<?, ?it/s]

...

Epoch 6/8 | train_acc=0.0752 | val_acc=0.0825 | val_f1=0.0473


  0%|          | 0/79 [00:00<?, ?it/s]

Epoch 7/8 | train_acc=0.1015 | val_acc=0.1201 | val_f1=0.0795


  0%|          | 0/79 [00:00<?, ?it/s]

Epoch 8/8 | train_acc=0.1362 | val_acc=0.1487 | val_f1=0.1033


In [27]:
lstm_loss, lstm_acc, lstm_f1, lstm_pred_ids = eval_model(lstm_model, test_loader, criterion, DEVICE)
lstm_preds = label_encoder.inverse_transform(np.array(lstm_pred_ids))
print("BiLSTM Accuracy:", round(lstm_acc, 4))
print("BiLSTM Weighted F1:", round(lstm_f1, 4))

BiLSTM Accuracy: 0.1487
BiLSTM Weighted F1: 0.1033


## 6. Transformer fine-tuning with DistilBERT

In [29]:
train_hf = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
test_hf = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

In [30]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenized_batch(batch):
    return tokenizer(batch["text"], ttruncation=True, padding="max_length", max_length=64)

train_tok = train_hf.map(tokenized_batch, batched=True)
test_tok = test_hf.map(tokenized_batch, batched=True)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/10003 [00:00<?, ? examples/s]

Map:   0%|          | 0/3080 [00:00<?, ? examples/s]

In [31]:
train_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

In [32]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy" : metric_acc.compute(predictions=preds, references=labels)["accuracy"],
        "f1_weighted": metric_f1.compute(predictions=preds, references=labels, average="weighted")["f1"]
    }


transformer_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [34]:
training_args = TrainingArguments(
    output_dir=str(MODEL_DIR / "distilbert_banking77"),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=transformer_model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [35]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,2.160738,1.946862,0.690584,0.652446
2,0.927651,0.866914,0.848377,0.840420
3,0.529608,0.545189,0.892857,0.891766
4,0.363556,0.426586,0.909091,0.909014
5,0.274362,0.396519,0.913636,0.913522


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=3130, training_loss=1.1565568719809047, metrics={'train_runtime': 236.1887, 'train_samples_per_second': 211.759, 'train_steps_per_second': 13.252, 'total_flos': 835159268108832.0, 'train_loss': 1.1565568719809047, 'epoch': 5.0})

In [36]:
transformer_metrics = trainer.evaluate()
transformer_preds_obj = trainer.predict(test_tok)

transformer_pred_ids = np.argmax(transformer_preds_obj.predictions, axis=1)
transformer_pred_labels = [id2label[i] for i in transformer_pred_ids]

In [37]:
transformer_acc = accuracy_score(test_df["label_name"], transformer_pred_labels)
transformer_f1 = f1_score(test_df["label_name"], transformer_pred_labels, average="weighted")


comparison = pd.DataFrame([
    {"model": "TF-IDF + LogisticRegression", "accuracy": tfidf_acc, "f1_weighted": tfidf_f1},
    {"model": "Word2Vec Avg + LogisticRegression", "accuracy": w2v_acc, "f1_weighted": w2v_f1},
    {"model": "BiLSTM + FastText init", "accuracy": lstm_acc, "f1_weighted": lstm_f1},
    {"model": "DistilBERT fine-tuned", "accuracy": transformer_acc, "f1_weighted": transformer_f1},
]).sort_values("f1_weighted", ascending=False)

comparison

,model,accuracy,f1_weighted
3,DistilBERT fine-tuned,0.913636,0.913522
0,TF-IDF + LogisticRegression,0.852273,0.852648
1,Word2Vec Avg + LogisticRegression,0.583117,0.572190
2,BiLSTM + FastText init,0.148701,0.103320


## 7. Error analysis

In [38]:
error_df = test_df.copy()
error_df["tfidf_pred"] = tfidf_preds
error_df["lstm_pred"] = lstm_preds
error_df["transformer_pred"] = transformer_pred_labels

transformer_errors = error_df[error_df["label_name"] != error_df["transformer_pred"]].copy()
transformer_errors[["text", "label_name", "transformer_pred"]].head(10)

,text,label_name,transformer_pred
0,How do I locate my card?,card_arrival,card_linking
3,Is there a way to know when my card will arrive?,card_arrival,card_delivery_estimate
5,When will I get my card?,card_arrival,card_delivery_estimate
11,How long does a card delivery take?,card_arrival,card_delivery_estimate
32,How do I know when my card will arrive?,card_arrival,card_delivery_estimate
138,Why am I being charged more ?,card_payment_wrong_exchange_rate,card_payment_fee_charged
150,the conversion value for my card payments is i...,card_payment_wrong_exchange_rate,cancel_transfer
156,How can I check the exchange rate applied to m...,card_payment_wrong_exchange_rate,exchange_rate
173,Where did this fee come from?,extra_charge_on_statement,card_payment_fee_charged
232,"I tried to take money from my card, but it did...",pending_cash_withdrawal,transfer_not_received_by_recipient
